# 04 · Scoring a model on synthetic scenes

A trained model is scored on scenes it never trained on. Three numbers per scene:

* **class**: is its top class the true class?
* **pointing**: does the heatmap's brightest cell land inside the sign's box?
* **joint**: both at once. This is the project's headline number.

**val** is for looking as often as you like: choosing the best epoch, digging into mistakes. **test** is
scored once per trained model and then left alone. If you looked at test while tweaking, you would gradually
tune the model to it, and the score would stop meaning "how good is it on new scenes". The scripts enforce
this (the *test guard*), and this notebook only *reads* test results unless you turn on `SCORE_TEST`.

**You'll see:** a live val score, the hardest classes and the classes most often confused, pictures of mistakes,
and the test scores of every version.

**Needs:** a trained run and `data/processed/val.npz`. **Writes:** nothing, unless you turn on a flag.

In [ ]:
# ---- Settings: the only cell you need to edit ----
RUN_NAME = "peak_fresh_aug"  # which trained run to score (newest run with this name)
RUN_FOR_REAL = False  # True: record the val score to outputs/eval/<run>/best_val.json via scripts/evaluate.py
SCORE_TEST = False  # True: score test ONCE for this run (refused, with an error, if it was already scored)

In [ ]:
# Setup (the same in every notebook). All paths in this project (configs/, data/, outputs/) are relative
# to the repo root, and the scripts expect to run from there, so move this notebook's working folder up
# to the root first. Safe to run twice.
import os
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists():  # walk up until we reach the folder holding pyproject.toml
    if root == root.parent:
        raise RuntimeError("can't find the repo root: open this notebook from inside FindTheObject-CNN-Project")
    root = root.parent
os.chdir(root)
if str(root / "notebooks") not in sys.path:
    sys.path.insert(0, str(root / "notebooks"))  # lets us import notebooks/nbhelpers.py

from nbhelpers import NB_OUT, ROOT, find_run, read_json, require, run_script
print("working in", ROOT)

In [ ]:
# Stop here, with a clear message, if a file this notebook reads is missing; then find the run's checkpoint.
require("data/processed/val.npz", how="python scripts/make_dataset.py --config configs/baseline.yaml")
RUN_DIR = find_run(RUN_NAME)
CKPT = (RUN_DIR / "best.pt").relative_to(ROOT)  # relative, as the scripts record it in their JSON
print("checkpoint:", CKPT)

## 1. Score on all 5,000 val scenes

`evaluate` runs the model over the val set in batches of 256 and counts the outcomes. It is the same function
training calls after every epoch, so the score should match what the checkpoint recorded when it was saved.

In [ ]:
# Load the model, score it on val (~6 s on MPS), and compare with the score saved inside the checkpoint.
from torch.utils.data import DataLoader

from findobject.data.dataset import SignScenes
from findobject.eval.evaluate import evaluate
from findobject.training.checkpoint import load_checkpoint_info, load_model
from findobject.utils.device import pick_device

device = pick_device("auto")
info = load_checkpoint_info(CKPT)  # epoch, config and val metrics stored when best.pt was saved
model = load_model(CKPT, device)
val = SignScenes(ROOT / "data/processed/val.npz")
result = evaluate(model, DataLoader(val, batch_size=256), device, info.config["train"]["heatmap_loss_weight"])

print(f"best epoch {info.epoch}")
for key in ("joint_acc", "class_acc", "pointing_acc", "center_err_median", "center_err_mean"):
    print(f"{key:18} now {result.scalars()[key]:.4f}   stored {info.metrics[key]:.4f}")

`center_err` is the distance in pixels from the predicted peak cell's center to the sign's true center. The median
is about 1 px, so when the model finds the sign it finds its middle. Small last-digit differences from the
stored numbers can appear: the Mac GPU doesn't add numbers in exactly the same order every time.

## 2. Which classes are hard?

`class_report` counts, for each true class, how many scenes there were and what the model predicted. The
**confusion matrix** has one row per true class and one column per predicted class. Everything off the diagonal
is a mistake.

In [ ]:
# Accuracy for each of the 43 classes, worst first. n = how many val scenes that class has.
import matplotlib.pyplot as plt
import numpy as np

from findobject.data.class_names import CLASS_NAMES
from findobject.eval.report import class_report

report = class_report(result.y_true, result.y_pred, 43)
acc = np.array([a if a is not None else np.nan for a in report.accuracy])
order = np.argsort(acc)
plt.figure(figsize=(14, 3))
plt.bar(range(43), acc[order])
plt.xticks(range(43), [CLASS_NAMES[k] for k in order], rotation=90, fontsize=7)
plt.ylim(min(0.8, np.nanmin(acc) - 0.02), 1), plt.ylabel("val class accuracy"), plt.title("per-class accuracy, worst first")
plt.show()
for k in order[:5]:
    print(f"{CLASS_NAMES[k]:28} {acc[k]:.3f}  (n={report.counts[k]})")

In [ ]:
# The most common (true -> predicted) mistakes: the largest off-diagonal entries of the confusion matrix.
conf = np.array(report.confusion)
np.fill_diagonal(conf, 0)  # ignore correct answers
pairs = sorted(((conf[t, p], t, p) for t in range(43) for p in range(43) if conf[t, p]), reverse=True)[:8]
for count, t, p in pairs:
    print(f"{count:3} x  {CLASS_NAMES[t]:28} -> {CLASS_NAMES[p]}")

Most mistakes are between signs that look alike at 16–48 px: speed limits that differ by one digit, and
triangular warning signs whose small center pictures blur together.

## 3. Pictures of mistakes

`predict` turns the model's raw outputs into readable predictions, and `draw_prediction` draws one: the heatmap
glowing over the scene, a cyan ring at the peak, the true box in green, and the title in red with the true class
when the class is wrong.

In [ ]:
# Draw the first 12 val scenes the model got wrong (wrong class, or peak outside the box).
import torch

from findobject.inference.predict import predict
from findobject.viz.overlay import draw_prediction

wrong = torch.nonzero(~((result.y_true == result.y_pred) & result.hit)).flatten()[:12].tolist()
print(f"{int((~((result.y_true == result.y_pred) & result.hit)).sum())} of {len(val)} val scenes are wrong; showing {len(wrong)}")
scenes = [val[i] for i in wrong]
preds = predict(model, torch.stack([s.image for s in scenes]).to(device))
fig, axes = plt.subplots(2, 6, figsize=(16, 6.4))
for ax, s, pred in zip(axes.flat, scenes, preds):
    draw_prediction(ax, s.image, pred, true_box=tuple(s.box.tolist()), true_class=int(s.class_id))
plt.tight_layout()
plt.show()

## 4. Test scores of every version (read from disk, nothing re-scored)

Each version's test result was recorded once, when that version was finished. v1–v1.5b were scored by
`evaluate.py` (`best_test.json`). v3a and v3a.1 were scored by `evaluate_presence.py` (`presence_test.json`,
notebook 06), which records the same joint accuracy among its numbers.

Three seeds (the same recipe trained three times from different random starting weights) show how much a score
moves by luck alone. That spread is the yardstick for whether a change between versions is real.

In [ ]:
# One row per run that has a recorded test score.
EVAL = ROOT / "outputs/eval"
VERSION = {"baseline": "v1", "peak": "v1.5", "peak_fresh_aug": "v1.5b", "peak_empty": "v3a", "peak_decoy": "v3a.1"}
print(f"{'version':8} {'run':34} {'test joint':>10}")
for name, version in VERSION.items():
    for seed_suffix in ("", "_s1", "_s2"):
        try:
            run = find_run(name + seed_suffix).name
        except FileNotFoundError:
            continue  # this version has no run with that seed
        if (EVAL / run / "best_test.json").exists():
            joint = read_json(EVAL / run / "best_test.json")["metrics"]["joint_acc"]
        elif (EVAL / run / "presence_test.json").exists():
            joint = read_json(EVAL / run / "presence_test.json")["report"]["all"]["joint"]["value"]
        else:
            continue  # trained but never scored on test
        print(f"{version:8} {run:34} {joint:10.4f}")

## Run it for real: record the scores

* `--split val` records the val score above to `outputs/eval/<run>/best_val.json`, with per-class numbers, the
  confusion matrix and per-scene outcomes. Safe to repeat.
* `--split test` scores the test set **once per run**. If this run's test score is already recorded (every
  run above already has one), the script refuses with exit code 2 and this cell raises an error. That's the
  test guard working, not a bug. The script's `--force` flag overrides it; it's left out here on purpose.

In [ ]:
# Record the val score. Only runs when RUN_FOR_REAL = True.
run_script("evaluate", "--checkpoint", CKPT, "--split", "val", enabled=RUN_FOR_REAL)

In [ ]:
# Score the test set (once per run). Only runs when SCORE_TEST = True.
run_script("evaluate", "--checkpoint", CKPT, "--split", "test", enabled=SCORE_TEST)

**Next:** `05_real_photos.ipynb`: does a model trained on pasted signs work on real road photos?